In [ ]:
import os


for dirname, subdirs, filenames in os.walk('/kaggle/input'):
    if 'pre-event' in subdirs:
        print(dirname)

In [ ]:
import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

# Define exact paths
TRAIN_PATH = "/kaggle/input/datasets/anumegha/galaxeye/train/train"
VAL_PATH   = "/kaggle/input/datasets/anumegha/galaxeye/change_detection_assignment_data-20260506T095257Z-3-001/change_detection_assignment_data/val/val"
TEST_PATH  = "/kaggle/input/datasets/anumegha/galaxeye/change_detection_assignment_data-20260506T095257Z-3-001/change_detection_assignment_data/test/test"

# Count files
for name, path in [("Train", TRAIN_PATH), ("Val", VAL_PATH), ("Test", TEST_PATH)]:
    count = len(os.listdir(os.path.join(path, "target")))
    print(f"{name}: {count} samples")

# Check class imbalance across multiple train masks
print("\nChecking class distribution across 10 random train masks...")
mask_files = os.listdir(os.path.join(TRAIN_PATH, "target"))[:10]

total_change = 0
total_pixels = 0

for f in mask_files:
    mask = np.array(Image.open(os.path.join(TRAIN_PATH, "target", f)))
    total_change += np.sum(mask == 1)
    total_pixels += mask.size

print(f"Change pixels:    {total_change/total_pixels*100:.2f}%")
print(f"No-change pixels: {(total_pixels-total_change)/total_pixels*100:.2f}%")

# Visualise one sample
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

sample_name = mask_files[0]

pre  = np.array(Image.open(os.path.join(TRAIN_PATH, "pre-event",  sample_name)))
post = np.array(Image.open(os.path.join(TRAIN_PATH, "post-event", sample_name)))
mask = np.array(Image.open(os.path.join(TRAIN_PATH, "target",     sample_name)))

axes[0].imshow(pre)
axes[0].set_title("Pre-event (EO)")
axes[1].imshow(post, cmap="gray")
axes[1].set_title("Post-event (SAR)")
axes[2].imshow(mask, cmap="gray")
axes[2].set_title("Target Mask")

plt.tight_layout()
plt.show()

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import random

class ChangeDetectionDataset(Dataset):
    def __init__(self, base_path, patch_size=256, augment=False):
        self.base_path = base_path
        self.patch_size = patch_size
        self.augment = augment
        self.filenames = sorted(os.listdir(os.path.join(base_path, "target")))
    
    def __len__(self):
        return len(self.filenames)
    
    def __getitem__(self, idx):
        fname = self.filenames[idx]
        
        # Load pre-event (EO) - RGB
        pre = np.array(Image.open(os.path.join(self.base_path, "pre-event", fname)))
        # Load post-event (SAR) - grayscale, expand to 3rd dim
        post = np.array(Image.open(os.path.join(self.base_path, "post-event", fname)))
        if post.ndim == 2:
            post = post[:, :, np.newaxis]  # (H, W) -> (H, W, 1)
        # Load mask
        mask = np.array(Image.open(os.path.join(self.base_path, "target", fname)))
# label remapping as specified in assignment
# Background(0) + Intact(1) → No-Change(0)
# Damaged(2) + Destroyed(3) → Change(1)
        remapped = np.zeros_like(mask)
        remapped[mask == 2] = 1
        remapped[mask == 3] = 1
        mask = remapped
        
        # Random crop to patch_size
        h, w = mask.shape
        top  = random.randint(0, h - self.patch_size)
        left = random.randint(0, w - self.patch_size)
        
        pre  = pre [top:top+self.patch_size, left:left+self.patch_size]
        post = post[top:top+self.patch_size, left:left+self.patch_size]
        mask = mask[top:top+self.patch_size, left:left+self.patch_size]
        
        # Augmentation - random horizontal flip
        if self.augment and random.random() > 0.5:
            pre  = np.fliplr(pre ).copy()
            post = np.fliplr(post).copy()
            mask = np.fliplr(mask).copy()
        
        # Normalise to [0, 1] and convert to tensors
        pre  = torch.tensor(pre,  dtype=torch.float32).permute(2, 0, 1) / 255.0
        post = torch.tensor(post, dtype=torch.float32).permute(2, 0, 1) / 255.0
        mask = torch.tensor(mask, dtype=torch.long)
        
        # Concatenate pre (3ch) + post (1ch) = 4 channels total
        image = torch.cat([pre, post], dim=0)  # (4, 256, 256)
        
        return image, mask

# Test the dataset
train_dataset = ChangeDetectionDataset(TRAIN_PATH, patch_size=256, augment=True)
val_dataset   = ChangeDetectionDataset(VAL_PATH,   patch_size=256, augment=False)
test_dataset  = ChangeDetectionDataset(TEST_PATH,  patch_size=256, augment=False)

#  sample
image, mask = train_dataset[0]
print("Image shape:", image.shape)  # Should be (4, 256, 256)
print("Mask shape:",  mask.shape)   # Should be (256, 256)
print("Unique mask values:", torch.unique(mask))
print("Dataset sizes:", len(train_dataset), len(val_dataset), len(test_dataset))

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Train batches:", len(train_loader))
print("Val batches:",   len(val_loader))
print("Test batches:",  len(test_loader))

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

class ConvBlock(nn.Module):
    """Two convolution layers with batch normalisation and ReLU"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.block(x)

class UNetWithResNet(nn.Module):
    def __init__(self, in_channels=4, num_classes=2):
        super().__init__()
        
        # Load pretrained ResNet34 as encoder
        resnet = models.resnet34(weights='IMAGENET1K_V1')
        
        # Modify first layer to accept 4 channels instead of 3
        # We copy the pretrained RGB weights and average them for the 4th channel
        old_conv = resnet.conv1
        new_conv = nn.Conv2d(4, 64, kernel_size=7, stride=2, padding=3, bias=False)
        with torch.no_grad():
            new_conv.weight[:, :3] = old_conv.weight
            new_conv.weight[:, 3]  = old_conv.weight.mean(dim=1)
        resnet.conv1 = new_conv
        
        # Encoder stages from ResNet
        self.enc1 = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu)  # 64ch
        self.pool  = resnet.maxpool
        self.enc2  = resnet.layer1  # 64ch
        self.enc3  = resnet.layer2  # 128ch
        self.enc4  = resnet.layer3  # 256ch
        self.enc5  = resnet.layer4  # 512ch
        
        # Decoder stages
        self.dec4 = ConvBlock(512 + 256, 256)
        self.dec3 = ConvBlock(256 + 128, 128)
        self.dec2 = ConvBlock(128 + 64,  64)
        self.dec1 = ConvBlock(64  + 64,  64)
        
        # Final output layer
        self.final = nn.Conv2d(64, num_classes, kernel_size=1)
        
        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
    
    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)       # (B, 64,  H/2,  W/2)
        e2 = self.enc2(self.pool(e1))  # (B, 64,  H/4,  W/4)
        e3 = self.enc3(e2)      # (B, 128, H/8,  W/8)
        e4 = self.enc4(e3)      # (B, 256, H/16, W/16)
        e5 = self.enc5(e4)      # (B, 512, H/32, W/32)
        
        # Decoder with skip connections
        d4 = self.dec4(torch.cat([self.up(e5), e4], dim=1))
        d3 = self.dec3(torch.cat([self.up(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up(d2), e1], dim=1))
        
        # Upsample back to original size
        out = self.up(self.final(d1))
        
        return out

# Test the model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = UNetWithResNet(in_channels=4, num_classes=2).to(device)

# Test with one batch
test_input = torch.randn(2, 4, 256, 256).to(device)
test_output = model(test_input)
print("Input shape:",  test_input.shape)
print("Output shape:", test_output.shape)  # Should be (2, 2, 256, 256)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

In [ ]:
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau

# ---- Loss Function ----
# We combine Dice Loss + CrossEntropy to handle class imbalance

class DiceLoss(nn.Module):
    def __init__(self, smooth=1):
        super().__init__()
        self.smooth = smooth
    
    def forward(self, predictions, targets):
        # Apply softmax to get probabilities
        predictions = torch.softmax(predictions, dim=1)
        # Take the change class (index 1)
        pred = predictions[:, 1]
        targ = (targets == 1).float()
        
        intersection = (pred * targ).sum()
        dice = (2 * intersection + self.smooth) / (pred.sum() + targ.sum() + self.smooth)
        return 1 - dice

class CombinedLoss(nn.Module):
    def __init__(self):
        super().__init__()
        # Weight class 1 (change) higher because it's rare (only 4.87%)
        weight = torch.tensor([1.0, 20.0]).to(device)
        self.ce   = nn.CrossEntropyLoss(weight=weight)
        self.dice = DiceLoss()
    
    def forward(self, predictions, targets):
        return self.ce(predictions, targets) + self.dice(predictions, targets)

# ---- Training Setup ----
model     = UNetWithResNet(in_channels=4, num_classes=2).to(device)
criterion = CombinedLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode='max', patience=3, factor=0.5)

# ---- Metric Functions ----
def compute_metrics(preds, targets):
    preds   = preds.cpu().numpy().flatten()
    targets = targets.cpu().numpy().flatten()
    
    tp = ((preds == 1) & (targets == 1)).sum()
    fp = ((preds == 1) & (targets == 0)).sum()
    fn = ((preds == 0) & (targets == 1)).sum()
    tn = ((preds == 0) & (targets == 0)).sum()
    
    precision = tp / (tp + fp + 1e-8)
    recall    = tp / (tp + fn + 1e-8)
    f1        = 2 * precision * recall / (precision + recall + 1e-8)
    iou       = tp / (tp + fp + fn + 1e-8)
    
    return precision, recall, f1, iou

# ---- Training Loop ----
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    
    for images, masks in loader:
        images = images.to(device)
        masks  = masks.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(loader)

# ---- Validation Loop ----
def validate(model, loader, criterion):
    model.eval()
    total_loss = 0
    all_preds  = []
    all_masks  = []
    
    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device)
            masks  = masks.to(device)
            
            outputs = model(images)
            loss    = criterion(outputs, masks)
            total_loss += loss.item()
            
            preds = torch.argmax(outputs, dim=1)
            all_preds.append(preds)
            all_masks.append(masks)
    
    all_preds = torch.cat(all_preds)
    all_masks = torch.cat(all_masks)
    
    precision, recall, f1, iou = compute_metrics(all_preds, all_masks)
    
    return total_loss / len(loader), precision, recall, f1, iou

# ---- Run Training ----
NUM_EPOCHS = 20
best_f1    = 0
best_model_path = "/kaggle/working/best_model.pth"

print("Starting training...")
print(f"{'Epoch':>5} | {'Train Loss':>10} | {'Val Loss':>8} | {'Precision':>9} | {'Recall':>6} | {'F1':>6} | {'IoU':>6}")
print("-" * 70)

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
    val_loss, precision, recall, f1, iou = validate(model, val_loader, criterion)
    
    scheduler.step(f1)
    
    # Save best model
    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.state_dict(), best_model_path)
        saved = "✓ saved"
    else:
        saved = ""
    
    print(f"{epoch:>5} | {train_loss:>10.4f} | {val_loss:>8.4f} | {precision:>9.4f} | {recall:>6.4f} | {f1:>6.4f} | {iou:>6.4f} {saved}")

print(f"\nBest F1: {best_f1:.4f}")
print(f"Model saved to: {best_model_path}")

In [ ]:
import os

model_path = "/kaggle/working/best_model.pth"

if os.path.exists(model_path):
    size = os.path.getsize(model_path) / (1024*1024)
    print(f"Model saved! Size: {size:.1f} MB")
else:
    print("Model file not found!")

In [ ]:
import os
import numpy as np
from PIL import Image

print("Checking all train masks for unexpected values...")
mask_dir = os.path.join(TRAIN_PATH, "target")
bad_files = []

for fname in os.listdir(mask_dir):
    mask = np.array(Image.open(os.path.join(mask_dir, fname)))
    unique_vals = np.unique(mask)
    if not all(v in [0, 1] for v in unique_vals):
        bad_files.append((fname, unique_vals))

if bad_files:
    print(f"Found {len(bad_files)} files with unexpected values:")
    for f, vals in bad_files[:5]:
        print(f"  {f}: {vals}")
else:
    print("All masks are clean (only 0 and 1)")

In [ ]:
import torch
import numpy as np
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Load best model
model = UNetWithResNet(in_channels=4, num_classes=2).to(device)
model.load_state_dict(torch.load("/kaggle/working/best_model.pth"))
model.eval()
print("Model loaded successfully!")

# Run evaluation on test set
all_preds = []
all_masks = []

with torch.no_grad():
    for images, masks in test_loader:
        images = images.to(device)
        outputs = model(images)
        preds = torch.argmax(outputs, dim=1)
        all_preds.append(preds.cpu())
        all_masks.append(masks.cpu())

all_preds = torch.cat(all_preds).numpy().flatten()
all_masks = torch.cat(all_masks).numpy().flatten()

# Compute metrics
precision, recall, f1, iou = compute_metrics(
    torch.tensor(all_preds), 
    torch.tensor(all_masks)
)

print("\n===== TEST SET RESULTS =====")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")
print(f"IoU       : {iou:.4f}")

# Confusion matrix
cm = confusion_matrix(all_masks, all_preds)
print("\nConfusion Matrix:")
print(cm)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No-Change', 'Change'],
            yticklabels=['No-Change', 'Change'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - Test Set')
plt.tight_layout()
plt.savefig('/kaggle/working/confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
def evaluate_full_images(model, base_path, device):
    model.eval()
    all_preds = []
    all_masks = []
    
    filenames = sorted(os.listdir(os.path.join(base_path, "target")))
    
    with torch.no_grad():
        for fname in filenames:
            # Load full images
            pre  = np.array(Image.open(os.path.join(base_path, "pre-event",  fname)))
            post = np.array(Image.open(os.path.join(base_path, "post-event", fname)))
            if post.ndim == 2:
                post = post[:, :, np.newaxis]
            mask = np.array(Image.open(os.path.join(base_path, "target", fname)))
            remapped = np.zeros_like(mask)
            remapped[mask == 2] = 1
            remapped[mask == 3] = 1
            mask = remapped
            
            # Resize to 512x512 for memory efficiency
            pre_img  = Image.fromarray(pre).resize((512, 512))
            post_img = Image.fromarray(post.squeeze()).resize((512, 512))
            mask_img = Image.fromarray(mask).resize((512, 512), Image.NEAREST)
            
            pre  = torch.tensor(np.array(pre_img),  dtype=torch.float32).permute(2,0,1) / 255.0
            post = torch.tensor(np.array(post_img), dtype=torch.float32).unsqueeze(0)   / 255.0
            image = torch.cat([pre, post], dim=0).unsqueeze(0).to(device)
            
            output = model(image)
            pred   = torch.argmax(output, dim=1).squeeze(0)
            
            all_preds.append(pred.cpu().numpy().flatten())
            all_masks.append(np.array(mask_img).flatten())
    
    all_preds = np.concatenate(all_preds)
    all_masks = np.concatenate(all_masks)
    
    tp = ((all_preds == 1) & (all_masks == 1)).sum()
    fp = ((all_preds == 1) & (all_masks == 0)).sum()
    fn = ((all_preds == 0) & (all_masks == 1)).sum()
    
    precision = tp / (tp + fp + 1e-8)
    recall    = tp / (tp + fn + 1e-8)
    f1        = 2 * precision * recall / (precision + recall + 1e-8)
    iou       = tp / (tp + fp + fn + 1e-8)
    
    return precision, recall, f1, iou, all_preds, all_masks

print("Evaluating on validation set (full images)...")
p, r, f1, iou, _, _ = evaluate_full_images(model, VAL_PATH, device)
print(f"Val   → Precision: {p:.4f} | Recall: {r:.4f} | F1: {f1:.4f} | IoU: {iou:.4f}")

print("\nEvaluating on test set (full images)...")
p, r, f1, iou, test_preds, test_masks = evaluate_full_images(model, TEST_PATH, device)
print(f"Test  → Precision: {p:.4f} | Recall: {r:.4f} | F1: {f1:.4f} | IoU: {iou:.4f}")

In [ ]:
def visualize_predictions(model, base_path, device, num_samples=5, split_name="Test"):
    model.eval()
    filenames = sorted(os.listdir(os.path.join(base_path, "target")))
    # Pick samples spread across the dataset
    indices = np.linspace(0, len(filenames)-1, num_samples, dtype=int)
    
    fig, axes = plt.subplots(num_samples, 4, figsize=(16, num_samples*4))
    cols = ['Pre-Event (EO)', 'Post-Event (SAR)', 'Ground Truth', 'Prediction']
    
    for ax, col in zip(axes[0], cols):
        ax.set_title(col, fontsize=12, fontweight='bold')
    
    with torch.no_grad():
        for row, idx in enumerate(indices):
            fname = filenames[idx]
            
            pre  = np.array(Image.open(os.path.join(base_path, "pre-event",  fname)))
            post = np.array(Image.open(os.path.join(base_path, "post-event", fname)))
            if post.ndim == 2:
                post = post[:, :, np.newaxis]
            mask = np.array(Image.open(os.path.join(base_path, "target", fname)))
            remapped = np.zeros_like(mask)
            remapped[mask == 2] = 1
            remapped[mask == 3] = 1
            mask = remapped
            
            # Resize for model
            pre_img  = Image.fromarray(pre).resize((512, 512))
            post_img = Image.fromarray(post.squeeze()).resize((512, 512))
            
            pre_t  = torch.tensor(np.array(pre_img),  dtype=torch.float32).permute(2,0,1) / 255.0
            post_t = torch.tensor(np.array(post_img), dtype=torch.float32).unsqueeze(0)   / 255.0
            image  = torch.cat([pre_t, post_t], dim=0).unsqueeze(0).to(device)
            
            output = model(image)
            pred   = torch.argmax(output, dim=1).squeeze(0).cpu().numpy()
            
            # Plot
            axes[row, 0].imshow(pre)
            axes[row, 1].imshow(post.squeeze(), cmap='gray')
            axes[row, 2].imshow(mask, cmap='gray', vmin=0, vmax=1)
            axes[row, 3].imshow(pred, cmap='gray', vmin=0, vmax=1)
            
            for ax in axes[row]:
                ax.axis('off')
            axes[row, 0].set_ylabel(f'Sample {idx+1}', fontsize=9)
    
    plt.suptitle(f'{split_name} Set — Qualitative Results', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'/kaggle/working/predictions_{split_name.lower()}.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved predictions_{split_name.lower()}.png")

# Generate for both splits
visualize_predictions(model, VAL_PATH,  device, num_samples=5, split_name="Val")
visualize_predictions(model, TEST_PATH, device, num_samples=5, split_name="Test")

In [ ]:
#loading only essentials
import os
import torch
import torch.nn as nn
import torchvision.models as models
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import random
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# ---- Paths ----
TRAIN_PATH = "/kaggle/input/datasets/anumegha/galaxeye/train/train"
VAL_PATH   = "/kaggle/input/datasets/anumegha/galaxeye/change_detection_assignment_data-20260506T095257Z-3-001/change_detection_assignment_data/val/val"
TEST_PATH  = "/kaggle/input/datasets/anumegha/galaxeye/change_detection_assignment_data-20260506T095257Z-3-001/change_detection_assignment_data/test/test"

# ---- Device ----
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ---- Dataset ----
class ChangeDetectionDataset(Dataset):
    def __init__(self, base_path, patch_size=256, augment=False):
        self.base_path = base_path
        self.patch_size = patch_size
        self.augment = augment
        self.filenames = sorted(os.listdir(os.path.join(base_path, "target")))
    
    def __len__(self):
        return len(self.filenames)
    
    def __getitem__(self, idx):
        fname = self.filenames[idx]
        pre  = np.array(Image.open(os.path.join(self.base_path, "pre-event",  fname)))
        post = np.array(Image.open(os.path.join(self.base_path, "post-event", fname)))
        if post.ndim == 2:
            post = post[:, :, np.newaxis]
        mask = np.array(Image.open(os.path.join(self.base_path, "target", fname)))
        remapped = np.zeros_like(mask)
        remapped[mask == 2] = 1
        remapped[mask == 3] = 1
        mask = remapped
        h, w = mask.shape
        top  = random.randint(0, h - self.patch_size)
        left = random.randint(0, w - self.patch_size)
        pre  = pre [top:top+self.patch_size, left:left+self.patch_size]
        post = post[top:top+self.patch_size, left:left+self.patch_size]
        mask = mask[top:top+self.patch_size, left:left+self.patch_size]
        if self.augment and random.random() > 0.5:
            pre  = np.fliplr(pre ).copy()
            post = np.fliplr(post).copy()
            mask = np.fliplr(mask).copy()
        pre  = torch.tensor(pre,  dtype=torch.float32).permute(2, 0, 1) / 255.0
        post = torch.tensor(post, dtype=torch.float32).permute(2, 0, 1) / 255.0
        mask = torch.tensor(mask, dtype=torch.long)
        image = torch.cat([pre, post], dim=0)
        return image, mask

# ---- Dataloaders ----
test_dataset = ChangeDetectionDataset(TEST_PATH, patch_size=256, augment=False)
val_dataset  = ChangeDetectionDataset(VAL_PATH,  patch_size=256, augment=False)
test_loader  = DataLoader(test_dataset, batch_size=8, shuffle=False, num_workers=2)
val_loader   = DataLoader(val_dataset,  batch_size=8, shuffle=False, num_workers=2)

# ---- Model ----
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.block(x)

class UNetWithResNet(nn.Module):
    def __init__(self, in_channels=4, num_classes=2):
        super().__init__()
        resnet = models.resnet34(weights='IMAGENET1K_V1')
        old_conv = resnet.conv1
        new_conv = nn.Conv2d(4, 64, kernel_size=7, stride=2, padding=3, bias=False)
        with torch.no_grad():
            new_conv.weight[:, :3] = old_conv.weight
            new_conv.weight[:, 3]  = old_conv.weight.mean(dim=1)
        resnet.conv1 = new_conv
        self.enc1 = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu)
        self.pool  = resnet.maxpool
        self.enc2  = resnet.layer1
        self.enc3  = resnet.layer2
        self.enc4  = resnet.layer3
        self.enc5  = resnet.layer4
        self.dec4  = ConvBlock(512 + 256, 256)
        self.dec3  = ConvBlock(256 + 128, 128)
        self.dec2  = ConvBlock(128 + 64,  64)
        self.dec1  = ConvBlock(64  + 64,  64)
        self.final = nn.Conv2d(64, num_classes, kernel_size=1)
        self.up    = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
    
    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)
        e5 = self.enc5(e4)
        d4 = self.dec4(torch.cat([self.up(e5), e4], dim=1))
        d3 = self.dec3(torch.cat([self.up(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up(d2), e1], dim=1))
        return self.up(self.final(d1))

# ---- Metrics ----
def compute_metrics(preds, targets):
    preds   = preds.cpu().numpy().flatten()
    targets = targets.cpu().numpy().flatten()
    tp = ((preds == 1) & (targets == 1)).sum()
    fp = ((preds == 1) & (targets == 0)).sum()
    fn = ((preds == 0) & (targets == 1)).sum()
    precision = tp / (tp + fp + 1e-8)
    recall    = tp / (tp + fn + 1e-8)
    f1        = 2 * precision * recall / (precision + recall + 1e-8)
    iou       = tp / (tp + fp + fn + 1e-8)
    return precision, recall, f1, iou

print("All definitions loaded successfully!")
print("Test samples:", len(test_dataset))
print("Val samples:",  len(val_dataset))